# S7 — Exploring the PushT demonstrations

I wanted to understand what a training example contains before moving on to policy training: what the agent observes, what an expert action represents, and how those actions fit together over time. I organized this notebook around those questions, from dataset metadata to individual demonstrations and action chunks.

The notebook does not currently contain saved outputs, so the notes explain what I was looking for and what the inspection code establishes. Numerical and visual conclusions need a completed run.

## 1. Getting the environment ready

I wanted to use the same Python 3.12 `.venv` as the training scripts so that exploration and training share the same LeRobot installation. The first cell checks the Python version and repository setup.

Install the notebook dependencies once in a terminal:

```bash
uv pip install --python .venv/bin/python jupyterlab ipykernel matplotlib pandas ipywidgets
```

Select the `.venv` kernel and restart it after installing packages.


In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the repository's Python 3.12 .venv kernel.")

repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "third_party/lerobot/pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from within the embodied-ai repository.")

print("Python:", sys.executable)
print("Repository:", repo_root)


## 2. What does one demonstration contain?

I wanted to see the dataset size, sampling rate, and feature definitions before interpreting any plots. The metadata table is where I check the shapes and meanings of the image, state, and action fields.

I use `lerobot/pusht` with PyAV for video decoding because of the TorchCodec compatibility issues in this environment. `EPISODE_ID` selects the demonstration used for the closer inspection below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from lerobot.datasets.lerobot_dataset import LeRobotDataset

REPO_ID = "lerobot/pusht"
EPISODE_ID = 0
VIDEO_BACKEND = "pyav"
DATASET_ROOT = None  # Set this if the dataset is already saved locally.
dataset = LeRobotDataset(REPO_ID, root=DATASET_ROOT, video_backend=VIDEO_BACKEND)
print(f"{REPO_ID}: {dataset.num_episodes:,} episodes, {dataset.num_frames:,} frames, {dataset.fps} FPS")
display(pd.DataFrame([
    {"feature": key, "dtype": spec["dtype"], "shape": spec["shape"], "names": spec.get("names")}
    for key, spec in dataset.features.items()
]))

## 3. How long are the demonstrations, and are the numbers usable?

I wanted to check whether demonstrations have similar lengths and whether missing or non-finite values could affect the analysis. The summary and histogram describe episode lengths, while the finite-value checks cover the numeric states and actions across the dataset.

I measure duration as `(frame_count - 1) / FPS`: the elapsed time between the first and last sample. I also check that timestamps increase within the selected episode and compare their median spacing with the expected frame interval. These checks give me a basis for interpreting the time-series plots; they do not establish demonstration quality on their own.


In [ ]:
numeric_keys = ["episode_index", "timestamp", "observation.state", "action"]
table = dataset.select_columns(numeric_keys).with_format("numpy")
episode_ids = np.asarray(table["episode_index"]).reshape(-1)
all_states = np.asarray(table["observation.state"])
all_actions = np.asarray(table["action"])
all_timestamps = np.asarray(table["timestamp"]).reshape(-1)
ids, counts = np.unique(episode_ids, return_counts=True)
episode_summary = pd.DataFrame({"episode_id": ids, "frames": counts,
                                "duration_s": (counts - 1) / dataset.fps})
display(episode_summary.describe())
print("Finite states:", np.isfinite(all_states).all(), "| Finite actions:", np.isfinite(all_actions).all())
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(episode_summary["duration_s"], bins=25, edgecolor="white")
ax.set(xlabel="Episode duration (s)", ylabel="Demonstrations", title="Demonstration lengths")
plt.show()

In [ ]:
episode_indices = np.flatnonzero(episode_ids == EPISODE_ID)
if not len(episode_indices):
    raise ValueError(f"Episode {EPISODE_ID} is unavailable; choose from {ids.tolist()}.")
states = all_states[episode_indices]
actions = all_actions[episode_indices]
timestamps = all_timestamps[episode_indices] - all_timestamps[episode_indices[0]]
assert np.isfinite(states).all() and np.isfinite(actions).all(), "Nonfinite episode values"
assert np.all(np.diff(timestamps) > 0), "Episode timestamps must increase"
print(f"Episode {EPISODE_ID}: {len(actions)} frames, {timestamps[-1]:.2f} s")
print("State shape:", states.shape, "| Action shape:", actions.shape)
print("Median frame interval:", np.median(np.diff(timestamps)), "s; expected:", 1 / dataset.fps)
sample = dataset[int(episode_indices[0])]
display(pd.DataFrame([{"feature": k, "shape": tuple(v.shape) if hasattr(v, "shape") else None,
                       "type": type(v).__name__} for k, v in sample.items()]))

## 4. What happens during a demonstration?

I wanted to connect the numeric fields to the scene. Six frames spread across the selected episode provide a first view of the interaction between the agent, the T-block, and the goal.

I keep the image preview separate from the state trajectory because the image pixels and stored coordinates should not be assumed to align directly. The preview is useful for inspecting scene changes, but six frames alone are too sparse to explain every movement.


In [ ]:
image_keys = [key for key, spec in dataset.features.items() if spec["dtype"] in ("video", "image")]
if not image_keys:
    raise ValueError("This dataset has no image features.")
IMAGE_KEY = image_keys[0]

def to_rgb(value):
    image = value.detach().cpu().numpy() if hasattr(value, "detach") else np.asarray(value)
    if image.ndim == 3 and image.shape[0] in (1, 3, 4):
        image = np.moveaxis(image, 0, -1)
    return np.clip(image, 0, 1) if image.dtype.kind == "f" else image

preview_offsets = np.unique(np.linspace(0, len(episode_indices) - 1, 6, dtype=int))
fig, axes = plt.subplots(1, len(preview_offsets), figsize=(18, 3), squeeze=False)
for ax, offset in zip(axes.flat, preview_offsets):
    ax.imshow(to_rgb(dataset[int(episode_indices[offset])][IMAGE_KEY]))
    ax.set_title(f"t = {timestamps[offset]:.2f} s")
    ax.axis("off")
fig.suptitle(f"Episode {EPISODE_ID} — {IMAGE_KEY}")
plt.tight_layout()
plt.show()

## 5. What is the relationship between state and action?

The distinction I wanted to understand is that `observation.state` records the agent's position, while `action` gives its commanded target position. Neither field directly describes the T-block's pose. This matters when interpreting a trajectory: the plotted path follows the agent, rather than the object being pushed.

I compare state and action over time and in the XY plane to inspect how the commanded targets relate to the agent's motion. The distance plot measures their separation at the same recorded frame. Without checking the control timing, I cannot interpret that separation as a delay-corrected tracking error.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for dim in range(states.shape[1]):
    axes[0].plot(timestamps, states[:, dim], label=f"state[{dim}]")
for dim in range(actions.shape[1]):
    axes[1].plot(timestamps, actions[:, dim], label=f"action[{dim}]")
axes[0].set(ylabel="Agent position", title=f"Episode {EPISODE_ID}")
axes[1].set(xlabel="Time (s)", ylabel="Target position")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(states[:, 0], states[:, 1], label="Agent")
axes[0].plot(actions[:, 0], actions[:, 1], alpha=0.6, label="Action targets")
axes[0].scatter(*states[0, :2], label="Start")
axes[0].scatter(*states[-1, :2], marker="x", label="End")
axes[0].set(xlabel="X", ylabel="Y", title="Agent path and commanded targets")
axes[0].set_aspect("equal", adjustable="box")
axes[0].legend()
tracking_distance = np.linalg.norm(actions[:, :2] - states[:, :2], axis=1)
axes[1].plot(timestamps, tracking_distance)
axes[1].set(xlabel="Time (s)", ylabel="Distance (dataset units)", title="Target-to-agent distance (same frame)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. How much do demonstrations differ?

I wanted to look beyond a single episode and see the range of paths a policy would encounter. I select up to four episodes spread across the dataset for the trajectory comparison, then use all episodes for the action-coordinate histograms.

These views answer different questions: the paths preserve the order of movement, while the histograms summarize where targets occur. The histograms alone cannot tell me which sequences of actions are typical or whether a demonstration succeeds.


In [ ]:
comparison_ids = ids[np.unique(np.linspace(0, len(ids) - 1, min(4, len(ids)), dtype=int))]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for episode_id in comparison_ids:
    path = all_states[episode_ids == episode_id]
    axes[0].plot(path[:, 0], path[:, 1], label=f"Episode {episode_id}")
axes[0].set(xlabel="Agent X", ylabel="Agent Y", title="Expert trajectories")
axes[0].set_aspect("equal", adjustable="box")
axes[0].legend()
for dim in range(all_actions.shape[1]):
    axes[1].hist(all_actions[:, dim], bins=40, alpha=0.5, label=f"action[{dim}]")
axes[1].set(xlabel="Target position", ylabel="Frames", title="Action distribution across all episodes")
axes[1].legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame(all_actions, columns=[f"action[{i}]" for i in range(all_actions.shape[1])]).describe())

## 7. What would an action chunk look like?

I wanted to make the training targets concrete by pairing one observation with the next 16 expert actions. The example shows the resulting action-array shape and a mask identifying any padded entries.

At an episode boundary, the code repeats the final valid action to fill the requested length. Setting `START_FRAME = len(actions) - 4` leaves four real actions followed by padding, making that behavior easy to inspect. This helped clarify the distinction between the fixed chunk length and the number of valid future targets. The example illustrates chunking; LeRobot's training sampler handles its own sampling rules.


In [ ]:
START_FRAME = 0
HORIZON = 16
if not 0 <= START_FRAME < len(actions) or HORIZON < 1:
    raise ValueError("Choose an existing start frame and a positive horizon.")
valid_actions = actions[START_FRAME:START_FRAME + HORIZON]
valid_length = len(valid_actions)
action_chunk = np.pad(valid_actions, ((0, HORIZON - valid_length), (0, 0)), mode="edge")
action_is_pad = np.arange(HORIZON) >= valid_length
chunk_observation = dataset[int(episode_indices[START_FRAME])]
print("Observation image:", tuple(chunk_observation[IMAGE_KEY].shape))
print("Action chunk:", action_chunk.shape, "| Padding mask:", action_is_pad)
display(pd.DataFrame(action_chunk, columns=[f"action[{i}]" for i in range(actions.shape[1])]).assign(is_pad=action_is_pad))

## 8. How does the motion look continuously?

I wanted a continuous view to complement the isolated frames and coordinate plots. The animation lets me inspect the order of movements and relate them back to the state and action signals.

Playback uses every second frame and stops after at most 60 displayed frames. Its timing accounts for that stride, but the clip may cover only part of the demonstration, so it cannot establish the outcome of the full episode.


In [ ]:
from matplotlib.animation import FuncAnimation

PLAYBACK_STRIDE = 2
MAX_PLAYBACK_FRAMES = 60
playback_offsets = np.arange(0, len(episode_indices), PLAYBACK_STRIDE)[:MAX_PLAYBACK_FRAMES]
playback_images = [to_rgb(dataset[int(episode_indices[i])][IMAGE_KEY]) for i in playback_offsets]
fig, ax = plt.subplots(figsize=(4, 4))
artist = ax.imshow(playback_images[0])
ax.axis("off")
title = ax.set_title("")

def update(frame):
    artist.set_data(playback_images[frame])
    title.set_text(f"Episode {EPISODE_ID} | t = {timestamps[playback_offsets[frame]]:.2f} s")
    return artist, title

animation = FuncAnimation(fig, update, frames=len(playback_images),
                          interval=1000 * PLAYBACK_STRIDE / dataset.fps, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))

## 9. What I take into the training stage

The main distinction from this exploration is between the visual scene, the agent's measured position, and its commanded target. Each provides a different view of the demonstration. In particular, an agent trajectory alone does not tell me whether the T-block reached the goal.

The action-chunk example also makes the episode boundary explicit: a fixed-length target array can contain padded entries, so its length is not necessarily the number of valid future actions.

Before training, I want to check the following:

- Run the notebook and record the observed episode lengths, value checks, and differences between trajectories.
- Inspect another episode and compare the image sequence with its state and action plots.
- Try a chunk near the episode boundary and inspect the padding mask.
- Map the dataset fields to policy inputs, action targets, and metadata, then relate the sampling setup to `horizon`, `n_obs_steps`, and `n_action_steps`.
- Keep whole demonstrations together when defining training and validation splits so neighboring frames cannot leak between them.

To assess task success, I still need to inspect the object and goal or evaluate a policy using the environment's success criterion.
